In [ ]:
"""
Single-file Advanced Lane Detection (Udacity-style)
==================================================
Processes video frames with:
  1) Camera calibration (cached to calibration.pkl)
  2) Undistortion
  3) Binary thresholding (gradient + color)
  4) Perspective warp (bird's-eye)
  5) Sliding windows lane pixel detection (fallback) + search around poly (tracking)
  6) Polynomial fit smoothing over time
  7) Curvature + vehicle offset
  8) Lane overlay + text
  9) Video output (MoviePy)

Expected paths (edit if needed):
  - camera_cal/calibration*.jpg
  - project_video.mp4 (and/or challenge_video.mp4, harder_challenge_video.mp4)

Install:
  pip install numpy opencv-python moviepy matplotlib

Run:
  python lane_detection_single.py

Outputs:
  output_videos/out_<video>.mp4
"""

import os
import glob
import pickle
import cv2
import numpy as np
from moviepy import VideoFileClip


# -----------------------------
# Settings (edit if needed)
# -----------------------------
CALIB_GLOB = os.path.join("camera_cal", "calibration*.jpg")
CALIB_CACHE = "calibration.pkl"

VIDEOS = [
    "project_video.mp4",
    "challenge_video.mp4",
    "harder_challenge_video.mp4",
]
OUTPUT_VIDEO_DIR = "output_videos"

# Perspective transform tuning (ratios for 1280x720 Udacity videos)
SRC_RATIOS = [
    (0.44, 0.65),  # top-left
    (0.56, 0.65),  # top-right
    (0.85, 0.95),  # bottom-right
    (0.15, 0.95),  # bottom-left
]
DST_X_OFFSET_RATIO = 0.25

# Sliding windows
NWINDOWS = 9
MARGIN = 110
MINPIX = 50

# Pixel-to-meter conversion (Udacity typical)
YM_PER_PIX = 30 / 720
XM_PER_PIX = 3.7 / 700


# -----------------------------
# Calibration
# -----------------------------
def calibrate_camera(calibration_images_glob, nx=9, ny=6):
    objp = np.zeros((ny * nx, 3), np.float32)
    objp[:, :2] = np.mgrid[0:nx, 0:ny].T.reshape(-1, 2)

    objpoints = []
    imgpoints = []

    images = glob.glob(calibration_images_glob)
    if not images:
        raise FileNotFoundError(f"No calibration images found for glob: {calibration_images_glob}")

    img_shape = None
    for fname in images:
        img = cv2.imread(fname)
        if img is None:
            continue
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img_shape = gray.shape[::-1]
        found, corners = cv2.findChessboardCorners(gray, (nx, ny), None)
        if found:
            objpoints.append(objp)
            criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
            corners_refined = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
            imgpoints.append(corners_refined)

    if not objpoints:
        raise RuntimeError("Calibration failed: no chessboard corners detected.")

    rms, mtx, dist, _, _ = cv2.calibrateCamera(objpoints, imgpoints, img_shape, None, None)
    return mtx, dist, rms


def save_calibration(path, mtx, dist):
    with open(path, "wb") as f:
        pickle.dump({"mtx": mtx, "dist": dist}, f)


def load_calibration(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["mtx"], data["dist"]


def undistort(img_bgr, mtx, dist):
    return cv2.undistort(img_bgr, mtx, dist, None, mtx)


# -----------------------------
# Thresholding
# -----------------------------
def _safe_scale_to_255(arr):
    m = float(np.max(arr))
    if m <= 1e-6:
        return np.zeros_like(arr, dtype=np.uint8)
    return np.uint8(255 * arr / m)


def abs_sobel_thresh(img_bgr, orient="x", ksize=3, thresh=(20, 100)):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    dx, dy = (1, 0) if orient == "x" else (0, 1)
    sobel = cv2.Sobel(gray, cv2.CV_64F, dx, dy, ksize=ksize)
    scaled = _safe_scale_to_255(np.absolute(sobel))
    binary = np.zeros_like(scaled, dtype=np.uint8)
    binary[(scaled >= thresh[0]) & (scaled <= thresh[1])] = 1
    return binary


def mag_thresh(img_bgr, ksize=3, thresh=(30, 100)):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    sx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=ksize)
    sy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=ksize)
    mag = np.sqrt(sx**2 + sy**2)
    scaled = _safe_scale_to_255(mag)
    binary = np.zeros_like(scaled, dtype=np.uint8)
    binary[(scaled >= thresh[0]) & (scaled <= thresh[1])] = 1
    return binary


def hls_s_thresh(img_bgr, thresh=(170, 255)):
    hls = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HLS)
    s = hls[:, :, 2]
    binary = np.zeros_like(s, dtype=np.uint8)
    binary[(s >= thresh[0]) & (s <= thresh[1])] = 1
    return binary


def lab_b_thresh(img_bgr, thresh=(155, 200)):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    b = lab[:, :, 2].astype(np.float32)
    b_scaled = _safe_scale_to_255(b)
    binary = np.zeros_like(b_scaled, dtype=np.uint8)
    binary[(b_scaled >= thresh[0]) & (b_scaled <= thresh[1])] = 1
    return binary


def combined_binary(img_bgr):
    gx = abs_sobel_thresh(img_bgr, orient="x", ksize=3, thresh=(20, 100))
    gm = mag_thresh(img_bgr, ksize=3, thresh=(30, 100))
    s = hls_s_thresh(img_bgr, thresh=(170, 255))
    b = lab_b_thresh(img_bgr, thresh=(155, 200))

    grad = (gx == 1) | (gm == 1)
    col = (s == 1) | (b == 1)

    out = np.zeros_like(gx, dtype=np.uint8)
    out[grad & col] = 1
    return out


# -----------------------------
# Perspective transform
# -----------------------------
def get_perspective_matrices(img_shape_hw, src_ratios=SRC_RATIOS, dst_x_offset_ratio=DST_X_OFFSET_RATIO):
    h, w = img_shape_hw
    src = np.float32([[w * x, h * y] for (x, y) in src_ratios])

    offset = w * dst_x_offset_ratio
    dst = np.float32([
        [offset, 0],
        [w - offset, 0],
        [w - offset, h],
        [offset, h],
    ])

    M = cv2.getPerspectiveTransform(src, dst)
    Minv = cv2.getPerspectiveTransform(dst, src)
    return M, Minv, src, dst


def warp(img, M):
    h, w = img.shape[:2]
    return cv2.warpPerspective(img, M, (w, h), flags=cv2.INTER_LINEAR)


# -----------------------------
# Lane finding + tracking
# -----------------------------
class Line:
    def __init__(self, history=8):
        self.detected = False
        self.recent_fits = []
        self.best_fit = None
        self.history = history

    def update(self, fit):
        self.recent_fits.append(fit)
        if len(self.recent_fits) > self.history:
            self.recent_fits.pop(0)
        self.best_fit = np.mean(self.recent_fits, axis=0)
        self.detected = True

    def reset(self):
        self.detected = False
        self.recent_fits = []
        self.best_fit = None


def find_lane_pixels(binary_warped, nwindows=NWINDOWS, margin=MARGIN, minpix=MINPIX):
    histogram = np.sum(binary_warped[binary_warped.shape[0] // 2:, :], axis=0)

    midpoint = histogram.shape[0] // 2
    leftx_base = int(np.argmax(histogram[:midpoint]))
    rightx_base = int(np.argmax(histogram[midpoint:]) + midpoint)

    window_height = binary_warped.shape[0] // nwindows

    nonzero = binary_warped.nonzero()
    nonzeroy = np.array(nonzero[0])
    nonzerox = np.array(nonzero[1])

    leftx_current = leftx_base
    rightx_current = rightx_base

    left_lane_inds = []
    right_lane_inds = []

    for window in range(nwindows):
        win_y_low = binary_warped.shape[0] - (window + 1) * window_height
        win_y_high = binary_warped.shape[0] - window * window_height

        win_xleft_low = leftx_current - margin
        win_xleft_high = leftx_current + margin

        win_xright_low = rightx_current - margin
        win_xright_high = rightx_current + margin

        good_left_inds = ((nonzeroy >= win_y_low) & (nonzeroy < win_y_high) &
                          (nonzerox >= win_xleft_low) & (nonzerox < win_xleft_high)).nonzero()[0]
        good_right_inds = ((nonzeroy >= win_y_low) & (nonzeroy < win_y_high) &
                           (nonzerox >= win_xright_low) & (nonzerox < win_xright_high)).nonzero()[0]

        left_lane_inds.append(good_left_inds)
        right_lane_inds.append(good_right_inds)

        if len(good_left_inds) > minpix:
            leftx_current = int(np.mean(nonzerox[good_left_inds]))
        if len(good_right_inds) > minpix:
            rightx_current = int(np.mean(nonzerox[good_right_inds]))

    left_lane_inds = np.concatenate(left_lane_inds) if left_lane_inds else np.array([])
    right_lane_inds = np.concatenate(right_lane_inds) if right_lane_inds else np.array([])

    leftx = nonzerox[left_lane_inds]
    lefty = nonzeroy[left_lane_inds]
    rightx = nonzerox[right_lane_inds]
    righty = nonzeroy[right_lane_inds]

    return leftx, lefty, rightx, righty


def search_around_poly(binary_warped, left_fit, right_fit, margin=80):
    nonzero = binary_warped.nonzero()
    nonzeroy = np.array(nonzero[0])
    nonzerox = np.array(nonzero[1])

    left_lane_inds = ((nonzerox > (left_fit[0] * nonzeroy**2 + left_fit[1] * nonzeroy + left_fit[2] - margin)) &
                      (nonzerox < (left_fit[0] * nonzeroy**2 + left_fit[1] * nonzeroy + left_fit[2] + margin)))
    right_lane_inds = ((nonzerox > (right_fit[0] * nonzeroy**2 + right_fit[1] * nonzeroy + right_fit[2] - margin)) &
                       (nonzerox < (right_fit[0] * nonzeroy**2 + right_fit[1] * nonzeroy + right_fit[2] + margin)))

    leftx = nonzerox[left_lane_inds]
    lefty = nonzeroy[left_lane_inds]
    rightx = nonzerox[right_lane_inds]
    righty = nonzeroy[right_lane_inds]
    return leftx, lefty, rightx, righty


def fit_polynomial(binary_warped, left_line, right_line):
    if left_line.best_fit is not None and right_line.best_fit is not None and left_line.detected and right_line.detected:
        leftx, lefty, rightx, righty = search_around_poly(binary_warped, left_line.best_fit, right_line.best_fit, margin=80)
        # If not enough pixels, fall back to sliding windows
        if len(leftx) < 1500 or len(rightx) < 1500:
            leftx, lefty, rightx, righty = find_lane_pixels(binary_warped)
    else:
        leftx, lefty, rightx, righty = find_lane_pixels(binary_warped)

    if len(leftx) < 500 or len(rightx) < 500:
        left_line.reset()
        right_line.reset()
        return None, None

    left_fit = np.polyfit(lefty, leftx, 2)
    right_fit = np.polyfit(righty, rightx, 2)

    left_line.update(left_fit)
    right_line.update(right_fit)
    return left_line.best_fit, right_line.best_fit


def measure_curvature_and_offset(img_shape, left_fit, right_fit):
    h, w = img_shape[:2]
    ploty = np.linspace(0, h - 1, h)

    leftx = left_fit[0] * ploty**2 + left_fit[1] * ploty + left_fit[2]
    rightx = right_fit[0] * ploty**2 + right_fit[1] * ploty + right_fit[2]

    ym = ploty * YM_PER_PIX
    leftxm = leftx * XM_PER_PIX
    rightxm = rightx * XM_PER_PIX

    left_fit_cr = np.polyfit(ym, leftxm, 2)
    right_fit_cr = np.polyfit(ym, rightxm, 2)

    y_eval = np.max(ym)
    left_curverad = ((1 + (2 * left_fit_cr[0] * y_eval + left_fit_cr[1])**2)**1.5) / np.absolute(2 * left_fit_cr[0])
    right_curverad = ((1 + (2 * right_fit_cr[0] * y_eval + right_fit_cr[1])**2)**1.5) / np.absolute(2 * right_fit_cr[0])

    curvature = 0.5 * (left_curverad + right_curverad)

    lane_center = 0.5 * (leftx[-1] + rightx[-1])
    vehicle_center = w / 2
    offset_m = (vehicle_center - lane_center) * XM_PER_PIX  # sign convention: + => vehicle right of lane center? depends on definition
    return curvature, offset_m


# -----------------------------
# Pipeline object
# -----------------------------
class LanePipeline:
    def __init__(self, mtx, dist):
        self.mtx = mtx
        self.dist = dist
        self.left_line = Line(history=8)
        self.right_line = Line(history=8)
        self.M = None
        self.Minv = None
        self._last_shape = None

    def process_frame(self, frame_rgb):
        # MoviePy frames are RGB uint8
        frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        # Undistort
        undist_bgr = undistort(frame_bgr, self.mtx, self.dist)

        # Binary threshold
        binary = combined_binary(undist_bgr)

        # Perspective (compute once per resolution)
        h, w = binary.shape[:2]
        if self.M is None or self.Minv is None or self._last_shape != (h, w):
            self.M, self.Minv, self.src, self.dst = get_perspective_matrices((h, w))
            self._last_shape = (h, w)

        warped = warp(binary, self.M)

        left_fit, right_fit = fit_polynomial(warped, self.left_line, self.right_line)
        if left_fit is None:
            # If lost, return undistorted frame (still looks okay)
            return cv2.cvtColor(undist_bgr, cv2.COLOR_BGR2RGB)

        # Create lane polygon in warped space
        ploty = np.linspace(0, h - 1, h)
        leftx = left_fit[0] * ploty**2 + left_fit[1] * ploty + left_fit[2]
        rightx = right_fit[0] * ploty**2 + right_fit[1] * ploty + right_fit[2]

        lane_warp = np.zeros((h, w, 3), dtype=np.uint8)
        pts_left = np.array([np.transpose(np.vstack([leftx, ploty]))])
        pts_right = np.array([np.flipud(np.transpose(np.vstack([rightx, ploty])))])
        pts = np.hstack((pts_left, pts_right)).astype(np.int32)
        cv2.fillPoly(lane_warp, [pts], (0, 255, 0))

        # Unwarp to camera view and overlay
        lane_unwarped = cv2.warpPerspective(lane_warp, self.Minv, (w, h))
        result = cv2.addWeighted(undist_bgr, 1.0, lane_unwarped, 0.3, 0)

        curvature, offset_m = measure_curvature_and_offset(result.shape, left_fit, right_fit)

        # Text overlay
        cv2.putText(result, f"Curvature: {curvature:8.1f} m", (40, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3, cv2.LINE_AA)
        cv2.putText(result, f"Curvature: {curvature:8.1f} m", (40, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2, cv2.LINE_AA)

        side = "left" if offset_m > 0 else "right"
        cv2.putText(result, f"Offset: {abs(offset_m):.2f} m {side} of center", (40, 110),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3, cv2.LINE_AA)
        cv2.putText(result, f"Offset: {abs(offset_m):.2f} m {side} of center", (40, 110),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 2, cv2.LINE_AA)

        return cv2.cvtColor(result, cv2.COLOR_BGR2RGB)


# -----------------------------
# Main
# -----------------------------
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def main():
    ensure_dir(OUTPUT_VIDEO_DIR)

    # Calibration (cached)
    if os.path.exists(CALIB_CACHE):
        mtx, dist = load_calibration(CALIB_CACHE)
        print(f"[OK] Loaded calibration from {CALIB_CACHE}")
    else:
        print("[INFO] Calibrating camera...")
        mtx, dist, rms = calibrate_camera(CALIB_GLOB, nx=9, ny=6)
        print(f"[OK] Calibration RMS error: {rms:.4f}")
        save_calibration(CALIB_CACHE, mtx, dist)
        print(f"[OK] Saved calibration to {CALIB_CACHE}")

    pipeline = LanePipeline(mtx, dist)

    for vid in VIDEOS:
        if not os.path.exists(vid):
            print(f"[SKIP] Missing video: {vid}")
            continue

        out_path = os.path.join(OUTPUT_VIDEO_DIR, f"out_{os.path.basename(vid)}")
        print(f"[RUN] {vid} -> {out_path}")

        clip = VideoFileClip(vid)
        out_clip = clip.image_transform(pipeline.process_frame)
        out_clip.write_videofile(out_path, audio=False)

    print("[DONE] All available videos processed.")


if __name__ == "__main__":
    main()